<a href="https://colab.research.google.com/github/muhammadrakayansyah-beep/AIF_RAKAH/blob/main/aif_p4_25120200002_muhammad_rakah_yansyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 4 Prediksi angka dan kategori dalam finance


Dosen **Haikal Shiddiq S.Kom., M.T.**  
Universitas Cakrawala / EKP2166 / Revisi 13 November 2026

**Peran Anda adalah analis finance.** Anda tidak perlu menghafal perintah Python. Untuk setiap langkah, baca tujuan, minta Gemini menjelaskan kode dengan bahasa sehari-hari, jalankan, lalu periksa angka dan tulis kesimpulan sendiri.

### Cara mulai dan bekerja
1. Buka di Google Colab. Pilih **File > Save a copy in Drive**. Beri nama sesuai sesi dan identitas: `AIF_P3_NIM_Nama` atau `AIF_P4_NIM_Nama` untuk AiFn2; `AIF_P3_AiFn1_KelompokXX` atau `AIF_P4_AiFn1_KelompokXX` untuk AiFn1.
2. Klik **Connect**. Jalankan sel kode dari atas ke bawah dengan tombol segitiga. Satu sel adalah satu bagian dokumen. Komputer cloud yang menjalankan kode disebut runtime.
3. Buka panel **Gemini** lewat ikon Gemini/kilauan jika tersedia. Pilih atau sebutkan sel yang ingin dibahas. Tampilan tombol dapat berbeda antarakun. Jangan meminta seluruh notebook ditulis ulang.
4. Sebelum tiap langkah, gunakan prompt: **“Pada notebook ini, jelaskan fungsi Langkah [nomor] untuk mahasiswa finance yang belum pernah coding. Sebutkan tujuan, data masuk, hasil yang muncul, satuan, dan cara membaca satu baris. Jangan ubah atau menjalankan kode dahulu.”** Jika konteks sel tidak terbaca, tempel hanya kode sel latihan yang dimaksud.
5. Cocokkan penjelasan Gemini dengan data. Pada kegiatan membuat kode, minta satu perubahan kecil lalu tinjau sebelum menjalankan. **Kode acuan tersedia tepat di bawah kegiatan**, sehingga kelas tetap berjalan jika Gemini tidak dapat diakses atau hasilnya berbeda.
6. Untuk mengisi jawaban, klik dua kali sel teks **Jawaban Anda**. Simpan output dan penjelasan. Perubahan input perlu diikuti menjalankan ulang sel; hasil lama tidak otomatis diperbarui.

**AiFn2:** praktik dan tugas individual. **AiFn1:** praktik/tugas P3–P4 berkelompok setelah pembentukan kelompok melalui voting dan arahan dosen. Semua anggota mencatat kontribusi dan siap menjelaskan hasil. **Kuis tetap individual tanpa materi, notebook, atau AI.** Jika kelompok belum terbentuk, mulai di salinan masing-masing lalu ikuti arahan pengumpulan dosen.

Data seluruhnya **fiktif untuk pembelajaran**. CPU standar cukup. Tidak perlu GPU, API key, langganan AI, pemasangan Python lokal, atau menghubungkan seluruh Drive. Notebook memuat data yang sama dengan CSV dalam paket, sehingga upload CSV bersifat opsional.

**Tujuan:** membedakan regresi dan klasifikasi, mengenali data masukan serta target, mencoba model sederhana, dan menjelaskan keterbatasannya. Detail model diperdalam pada P5–P6.

**Jadwal WIB:** AiFn2 19 November 2026 pukul 20.00–22.00, jeda 20.00–20.10; AiFn1 20 November 2026 pukul 10.00–11.30, jeda 10.00–10.03.

In [ ]:
# @title Identitas dan kelas
nama = "Muhammad Rakah Yansyah" # @param {type:"string"}
nim = "25120200002" # @param {type:"string"}
kelas = "AiFn2" # @param ["AiFn2", "AiFn1"]
print("Nama:", nama)
print("NIM:", nim, "| Kelas:", kelas)

Nama: Muhammad Rakah Yansyah
NIM: 25120200002 | Kelas: AiFn2


In [ ]:
# @title Persiapan cukup klik jalankan
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}".replace(",", "_").replace(".", ",").replace("_", "."))
print("Tempat kerja siap. Semua nominal sumber memakai rupiah per bulan.")
print("Pada tampilan ringkas, nominal dikonversi ke Rp juta per bulan dan diberi judul satuan.")

Tempat kerja siap. Semua nominal sumber memakai rupiah per bulan.
Pada tampilan ringkas, nominal dikonversi ke Rp juta per bulan dan diberi judul satuan.


## Langkah 1 Mengenali data historis dan tujuan prediksi
Kita tetap berada dalam konteks pengajuan pinjaman. Dataset H001–H024 adalah **24 catatan historis fiktif tambahan**, terpisah dari A001–A007 di P3. Kita tidak menambahkan riwayat pembayaran hasil tebakan ke pengajuan P3.

**Pertanyaan regresi:** berapa perkiraan pengeluaran rutin noncicilan per bulan untuk pemohon baru? Masukannya pendapatan bulanan. Target belajarnya pengeluaran rutin noncicilan yang tercatat pada pemohon historis. Hasil prediksi masih perlu klarifikasi dari pemohon.

**Pertanyaan klasifikasi:** kategori keterlambatan apa yang diperkirakan? Masukannya rasio cicilan saat pengajuan. Target belajarnya kejadian **terlambat minimal 30 hari dalam tiga bulan pertama setelah pencairan**. Kode 0 = tidak mengalami kejadian tersebut; kode 1 = mengalami kejadian tersebut. Kategori prediksi ini tidak sama dengan probabilitas gagal bayar atau persetujuan kredit.

Kolom `bagian` memisahkan 18 contoh latih dan 6 contoh uji. Kolom ID dan bagian tidak menjadi masukan model.

**Prompt Gemini:** “Jelaskan Langkah 1 untuk mahasiswa finance. Apa data masukan dan jawaban historis untuk setiap model? Mengapa riwayat masa depan tidak boleh dijadikan masukan saat pengajuan baru?”

In [ ]:
# @title Langkah 1 Buka data historis fiktif
data_historis = pd.DataFrame([{'id_historis': 'H001', 'pendapatan_bulanan': 5000000, 'cicilan_bulanan': 1000000, 'pengeluaran_rutin_non_cicilan': 3100000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H002', 'pendapatan_bulanan': 6000000, 'cicilan_bulanan': 1500000, 'pengeluaran_rutin_non_cicilan': 3800000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H003', 'pendapatan_bulanan': 7000000, 'cicilan_bulanan': 2100000, 'pengeluaran_rutin_non_cicilan': 4500000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H004', 'pendapatan_bulanan': 8000000, 'cicilan_bulanan': 2800000, 'pengeluaran_rutin_non_cicilan': 4900000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H005', 'pendapatan_bulanan': 9000000, 'cicilan_bulanan': 3600000, 'pengeluaran_rutin_non_cicilan': 5600000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H006', 'pendapatan_bulanan': 10000000, 'cicilan_bulanan': 4500000, 'pengeluaran_rutin_non_cicilan': 6000000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H007', 'pendapatan_bulanan': 11000000, 'cicilan_bulanan': 5500000, 'pengeluaran_rutin_non_cicilan': 6700000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H008', 'pendapatan_bulanan': 12000000, 'cicilan_bulanan': 3360000, 'pengeluaran_rutin_non_cicilan': 7100000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H009', 'pendapatan_bulanan': 4000000, 'cicilan_bulanan': 880000, 'pengeluaran_rutin_non_cicilan': 2600000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H010', 'pendapatan_bulanan': 6000000, 'cicilan_bulanan': 1920000, 'pengeluaran_rutin_non_cicilan': 3600000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H011', 'pendapatan_bulanan': 8000000, 'cicilan_bulanan': 3040000, 'pengeluaran_rutin_non_cicilan': 4800000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H012', 'pendapatan_bulanan': 10000000, 'cicilan_bulanan': 4200000, 'pengeluaran_rutin_non_cicilan': 6200000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H013', 'pendapatan_bulanan': 12000000, 'cicilan_bulanan': 5760000, 'pengeluaran_rutin_non_cicilan': 7000000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H014', 'pendapatan_bulanan': 14000000, 'cicilan_bulanan': 3640000, 'pengeluaran_rutin_non_cicilan': 8000000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H015', 'pendapatan_bulanan': 5000000, 'cicilan_bulanan': 1700000, 'pengeluaran_rutin_non_cicilan': 3300000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H016', 'pendapatan_bulanan': 7000000, 'cicilan_bulanan': 2520000, 'pengeluaran_rutin_non_cicilan': 4200000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H017', 'pendapatan_bulanan': 9000000, 'cicilan_bulanan': 3960000, 'pengeluaran_rutin_non_cicilan': 5300000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'latih'}, {'id_historis': 'H018', 'pendapatan_bulanan': 11000000, 'cicilan_bulanan': 2640000, 'pengeluaran_rutin_non_cicilan': 6500000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'latih'}, {'id_historis': 'H019', 'pendapatan_bulanan': 13000000, 'cicilan_bulanan': 5980000, 'pengeluaran_rutin_non_cicilan': 7400000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'uji'}, {'id_historis': 'H020', 'pendapatan_bulanan': 15000000, 'cicilan_bulanan': 4500000, 'pengeluaran_rutin_non_cicilan': 8500000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'uji'}, {'id_historis': 'H021', 'pendapatan_bulanan': 6000000, 'cicilan_bulanan': 1980000, 'pengeluaran_rutin_non_cicilan': 3900000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'uji'}, {'id_historis': 'H022', 'pendapatan_bulanan': 8000000, 'cicilan_bulanan': 2960000, 'pengeluaran_rutin_non_cicilan': 4600000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'uji'}, {'id_historis': 'H023', 'pendapatan_bulanan': 10000000, 'cicilan_bulanan': 4100000, 'pengeluaran_rutin_non_cicilan': 6300000, 'terlambat_30hari_dalam_3bulan': 0, 'bagian': 'uji'}, {'id_historis': 'H024', 'pendapatan_bulanan': 12000000, 'cicilan_bulanan': 3480000, 'pengeluaran_rutin_non_cicilan': 7300000, 'terlambat_30hari_dalam_3bulan': 1, 'bagian': 'uji'}])
data_historis["rasio_persen"] = data_historis["cicilan_bulanan"] / data_historis["pendapatan_bulanan"] * 100
data_latih = data_historis[data_historis["bagian"] == "latih"].copy()
data_uji = data_historis[data_historis["bagian"] == "uji"].copy()
display(data_historis.head(6))
print("Jumlah historis:",len(data_historis),"| Contoh untuk belajar:",len(data_latih),"| Contoh yang disisihkan:",len(data_uji))
print("Kode target keterlambatan: 0 = tidak terlambat >=30 hari; 1 = terlambat >=30 hari dalam tiga bulan pertama.")

,id_historis,pendapatan_bulanan,cicilan_bulanan,pengeluaran_rutin_non_cicilan,terlambat_30hari_dalam_3bulan,bagian,rasio_persen
0,H001,5000000,1000000,3100000,0,latih,"20,00"
1,H002,6000000,1500000,3800000,0,latih,"25,00"
2,H003,7000000,2100000,4500000,0,latih,"30,00"
3,H004,8000000,2800000,4900000,1,latih,"35,00"
4,H005,9000000,3600000,5600000,1,latih,"40,00"
5,H006,10000000,4500000,6000000,1,latih,"45,00"


Jumlah historis: 24 | Contoh untuk belajar: 18 | Contoh yang disisihkan: 6
Kode target keterlambatan: 0 = tidak terlambat >=30 hari; 1 = terlambat >=30 hari dalam tiga bulan pertama.


## Langkah 2 Meminta program mempelajari contoh historis
**Supervised learning** berarti belajar dari contoh yang memiliki masukan dan jawaban historis. Pada regresi, jawabannya angka rupiah. Pada klasifikasi, jawabannya kategori kejadian.

Program menggunakan regresi linear sederhana dan pohon keputusan sederhana. Anda cukup memahami apa yang masuk dan apa yang dipelajari. `fit` berarti mempelajari contoh latih. `predict` berarti menghasilkan perkiraan untuk masukan yang diberikan.

**Yang akan muncul:** konfirmasi bahwa dua model telah belajar dari 18 contoh. Enam contoh uji belum dipakai untuk belajar. Membuat rumus rasio saja belum termasuk melatih model.

**Prompt Gemini:** “Jelaskan tujuan Langkah 2, arti fit dan predict, serta mengapa data_uji tidak dipakai saat belajar. Jangan menguraikan rumus algoritma. Gunakan analogi analis mempelajari berkas lama lalu memeriksa berkas yang belum dilihat.”

In [ ]:
# @title Langkah 2 Belajar dari 18 contoh saja
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
model_nominal = LinearRegression()
model_kategori = DecisionTreeClassifier(max_depth=1, random_state=42)
model_nominal.fit(data_latih[["pendapatan_bulanan"]], data_latih["pengeluaran_rutin_non_cicilan"])
model_kategori.fit(data_latih[["rasio_persen"]], data_latih["terlambat_30hari_dalam_3bulan"])
print("Dua model telah belajar dari",len(data_latih),"contoh latih.")
print(len(data_uji),"contoh uji tetap terpisah. Tidak ada jawaban masa depan pada masukan model.")
def nama_kategori(kode):
    return "Terlambat >=30 hari" if int(kode)==1 else "Tidak terlambat >=30 hari"
def buat_ringkasan(pendapatan_juta, cicilan_juta, nama_skenario):
    if pendapatan_juta<=0 or cicilan_juta<0:
        raise ValueError("Pendapatan harus positif dan cicilan tidak negatif. Periksa angka formulir.")
    pendapatan = float(pendapatan_juta)*1e6
    cicilan = float(cicilan_juta)*1e6
    rasio = cicilan/pendapatan*100
    nominal = float(model_nominal.predict(pd.DataFrame({"pendapatan_bulanan":[pendapatan]}))[0])
    kategori = int(model_kategori.predict(pd.DataFrame({"rasio_persen":[rasio]}))[0])
    return pd.DataFrame({"Skenario":[nama_skenario],"Pendapatan Rp juta/bulan":[pendapatan_juta],"Cicilan Rp juta/bulan":[cicilan_juta],"Rasio cicilan %":[rasio],"Prediksi pengeluaran rutin Rp juta/bulan":[nominal/1e6],"Prediksi kategori":[nama_kategori(kategori)]})

Dua model telah belajar dari 18 contoh latih.
6 contoh uji tetap terpisah. Tidak ada jawaban masa depan pada masukan model.


## Langkah 3 Membaca prediksi awal dan mencoba skenario
**Kasus awal:** pemohon baru berpendapatan Rp8 juta per bulan, dengan cicilan Rp2,8 juta per bulan. Bandingkan dengan skenario pendapatan Rp10 juta dan cicilan tetap Rp2,8 juta.

Isi formulir dalam **juta rupiah per bulan**. Gunakan titik untuk desimal, misalnya `2.8`. Ini perubahan asumsi latihan, bukan perubahan bukti pendapatan pemohon. Jalankan ulang setelah mengubah angka.

**Cara membaca:** kolom rasio adalah hitungan dari input. Kolom prediksi pengeluaran adalah estimasi model dalam Rp juta/bulan. Kolom kategori berisi perkiraan kejadian, bukan perintah menerima atau menolak kredit. Tidak ada aturan manual 35% yang dipakai sebagai ketentuan bank.

**Prompt Gemini:** “Jelaskan satu baris hasil Langkah 3. Bedakan input, hitungan rasio, prediksi angka dan prediksi kategori. Mengapa prediksi pengeluaran perlu dikonfirmasi dan kategori belum membuktikan kejadian akan terjadi?”

In [ ]:
# @title Langkah 3 Formulir kasus awal dan skenario cukup ubah angka
pendapatan_awal_juta = 8.0 # @param {type:"number"}
cicilan_awal_juta = 2.8 # @param {type:"number"}
pendapatan_skenario_juta = 10.0 # @param {type:"number"}
cicilan_skenario_juta = 2.8 # @param {type:"number"}
hasil_skenario = pd.concat([buat_ringkasan(pendapatan_awal_juta,cicilan_awal_juta,"Awal"),buat_ringkasan(pendapatan_skenario_juta,cicilan_skenario_juta,"Skenario")],ignore_index=True)
display(hasil_skenario.round(2))
for _,baris in hasil_skenario.iterrows():
    print(f"{baris['Skenario']}: rasio {baris['Rasio cicilan %']:.2f}%; perkiraan pengeluaran rutin Rp{baris['Prediksi pengeluaran rutin Rp juta/bulan']:.2f} juta/bulan; kategori {baris['Prediksi kategori']}.")
print("Skenario hanya mengubah asumsi. Prediksi bukan kepastian atau persetujuan kredit.")

,Skenario,Pendapatan Rp juta/bulan,Cicilan Rp juta/bulan,Rasio cicilan %,Prediksi pengeluaran rutin Rp juta/bulan,Prediksi kategori
0,Awal,"8,00","2,80","35,00","4,87",Terlambat >=30 hari
1,Skenario,"10,00","2,80","28,00","5,98",Tidak terlambat >=30 hari


Awal: rasio 35.00%; perkiraan pengeluaran rutin Rp4.87 juta/bulan; kategori Terlambat >=30 hari.
Skenario: rasio 28.00%; perkiraan pengeluaran rutin Rp5.98 juta/bulan; kategori Tidak terlambat >=30 hari.
Skenario hanya mengubah asumsi. Prediksi bukan kepastian atau persetujuan kredit.


### Jawaban Anda untuk Langkah 3
Catat hasil awal dan skenario, satuan setiap angka, serta perubahan rasio dalam **poin persentase**. Jelaskan alasan perubahan prediksi dan informasi tambahan yang diperlukan sebelum menyarankan tindakan kepada pemohon.

Jawaban:

| | Input pendapatan (Rp juta/bln) | Input cicilan (Rp juta/bln) | Rasio cicilan (%) | Prediksi pengeluaran rutin (Rp juta/bln) | Prediksi kategori |
|---|---|---|---|---|---|
| Awal | 8 | 2,8 | 2,8 ÷ 8 × 100 = **35%** | **4,87** | Terlambat >=30 hari |
| Skenario | 10 | 2,8 | 2,8 ÷ 10 × 100 = **28%** | **5,98** | Tidak terlambat >=30 hari |

- **Perubahan rasio:** 35% − 28% = turun **7 poin persentase**. Penurunannya bukan 7%, karena yang dikurangkan adalah dua angka persen.
- **Alasan prediksi berubah:** model kategori (pohon keputusan) belajar dari 18 contoh latih bahwa rasio di atas sekitar 33% cenderung masuk kategori terlambat. Rasio 35% berada di atas batas itu, sedangkan 28% di bawahnya, jadi kategorinya berbalik. Model nominal (regresi linear) belajar bahwa pengeluaran rutin naik seiring pendapatan, sekitar Rp0,55 juta per tambahan Rp1 juta pendapatan. Pendapatan naik Rp2 juta, jadi prediksi pengeluaran naik dari Rp4,87 juta ke Rp5,98 juta.
- **Satuan:** input dan prediksi pengeluaran dalam Rp juta per bulan, rasio dalam persen, sedangkan kategori tidak bersatuan (berupa label).
- **Informasi tambahan sebelum menyarankan tindakan:** bukti pendapatan (slip gaji atau mutasi rekening), pengeluaran rutin aktual dari pemohon (karena 4,87 dan 5,98 juta hanya estimasi), jumlah tanggungan, kewajiban lain, dan riwayat pembayaran. Skenario Rp10 juta hanyalah asumsi, bukan bukti pendapatan.

## Langkah 4 Membandingkan prediksi dengan jawaban pada contoh uji
**Pertanyaan finance:** apakah perkiraan model cocok pada semua berkas yang belum dipakai untuk belajar?

Enam contoh uji sudah memiliki jawaban historis, tetapi baru dibuka untuk pemeriksaan setelah model belajar. Kolom aktual adalah kejadian yang tercatat pada data fiktif. Kolom prediksi adalah keluaran model. Cocok berarti keduanya sama pada baris itu.

**Cara membaca:** jumlah cocok dari enam contoh hanya hasil latihan kecil ini. Angka tersebut tidak membuktikan keandalan untuk nasabah nyata. Pembahasan metrik evaluasi lebih lengkap ada pada sesi 10.

**Prompt Gemini:** “Jelaskan Langkah 4 dan satu contoh hasil yang tidak cocok. Apa dampak jika model memprediksi terlambat padahal aktual tidak terlambat, atau sebaliknya? Jangan mengubah model untuk mengejar semua jawaban uji.”

In [ ]:
# @title Langkah 4 Buka hasil uji setelah model belajar
pred_uji_nominal = model_nominal.predict(data_uji[["pendapatan_bulanan"]])
pred_uji_kategori = model_kategori.predict(data_uji[["rasio_persen"]])
tabel_uji = pd.DataFrame({"ID":data_uji["id_historis"].to_list(),"Aktual pengeluaran Rp juta":(data_uji["pengeluaran_rutin_non_cicilan"]/1e6).to_list(),"Prediksi pengeluaran Rp juta":pred_uji_nominal/1e6,"Aktual kategori":[nama_kategori(x) for x in data_uji["terlambat_30hari_dalam_3bulan"]],"Prediksi kategori":[nama_kategori(x) for x in pred_uji_kategori]})
tabel_uji["Kategori cocok"] = ["Ya" if a==b else "Tidak" for a,b in zip(data_uji["terlambat_30hari_dalam_3bulan"],pred_uji_kategori)]
display(tabel_uji.round(2))
jumlah_cocok = int(sum(tabel_uji["Kategori cocok"]=="Ya"))
print("Prediksi kategori cocok:",jumlah_cocok,"dari",len(data_uji),"contoh uji.")
print("Perbedaan nominal menunjukkan estimasi masih memiliki selisih. Ini bukti terbatas pada data fiktif, bukan peluang gagal bayar seorang pemohon.")

,ID,Aktual pengeluaran Rp juta,Prediksi pengeluaran Rp juta,Aktual kategori,Prediksi kategori,Kategori cocok
0,H019,"7,40","7,64",Terlambat >=30 hari,Terlambat >=30 hari,Ya
1,H020,"8,50","8,75",Tidak terlambat >=30 hari,Tidak terlambat >=30 hari,Ya
2,H021,"3,90","3,76",Tidak terlambat >=30 hari,Tidak terlambat >=30 hari,Ya
3,H022,"4,60","4,87",Terlambat >=30 hari,Terlambat >=30 hari,Ya
4,H023,"6,30","5,98",Tidak terlambat >=30 hari,Terlambat >=30 hari,Tidak
5,H024,"7,30","7,08",Terlambat >=30 hari,Tidak terlambat >=30 hari,Tidak


Prediksi kategori cocok: 4 dari 6 contoh uji.
Perbedaan nominal menunjukkan estimasi masih memiliki selisih. Ini bukti terbatas pada data fiktif, bukan peluang gagal bayar seorang pemohon.


## Langkah 5 Meminta Gemini menyusun tabel perbandingan
**Tujuan finance:** menyajikan dua asumsi agar analis mudah membandingkan hasil. Gemini membantu menulis panggilan fungsi, sedangkan Anda memeriksa satuan dan maknanya.

Salin prompt berikut ke panel Gemini, tinjau kode, lalu tempel di sel kosong berikut. Kode acuan tersedia jika akses atau jawaban Gemini terkendala.

> Pada notebook ini tersedia fungsi buat_ringkasan(pendapatan_juta, cicilan_juta, nama_skenario). Buat SATU sel kode yang memanggil fungsi itu untuk kondisi (8, 2.8, "Awal") dan (10, 2.8, "Skenario"), lalu gabungkan hasil menjadi hasil_gemini_p4 dan tampilkan. Jangan melatih ulang model, mengubah data latih/uji, membuat keputusan kredit, memasang paket, atau mengakses internet. Sebelum menjalankan, jelaskan bahwa rasio adalah hitungan dan dua kolom prediksi adalah hasil model.

In [ ]:
# Tempel kode Gemini Langkah 5 setelah ditinjau.
# Jika tidak tersedia, gunakan kode acuan di bawah.
# Catatan: pada versi ini Langkah 5 dikerjakan memakai kode acuan (Gemini belum dipakai).

In [ ]:
# @title Kode acuan Langkah 5 jika diperlukan
hasil_acuan_p4 = pd.concat([buat_ringkasan(8,2.8,"Awal"),buat_ringkasan(10,2.8,"Skenario")],ignore_index=True)
display(hasil_acuan_p4.round(2))
print("Data dan model tidak berubah. Yang dibandingkan hanya dua masukan pemohon fiktif.")
if "hasil_gemini_p4" in globals():
    print("Bandingkan tabel Gemini dan tabel acuan, termasuk judul satuan serta kategori.")

,Skenario,Pendapatan Rp juta/bulan,Cicilan Rp juta/bulan,Rasio cicilan %,Prediksi pengeluaran rutin Rp juta/bulan,Prediksi kategori
0,Awal,8,"2,80","35,00","4,87",Terlambat >=30 hari
1,Skenario,10,"2,80","28,00","5,98",Tidak terlambat >=30 hari


Data dan model tidak berubah. Yang dibandingkan hanya dua masukan pemohon fiktif.


## Langkah 6 Menulis memo analis dan mengumpulkan
### Jawaban Anda
1. Jelaskan tujuan Langkah 1 dan Langkah 2. Bedakan masukan, target historis, dan prediksi.
2. Mengapa regresi cocok untuk perkiraan nominal pengeluaran dan klasifikasi cocok untuk kategori keterlambatan?
3. Catat input awal/skenario, rasio, estimasi nominal, dan kategori. Apa yang berubah? Berapa perubahan rasio dalam poin persentase?
4. Sebutkan ID contoh uji yang kategori prediksinya tidak cocok. Jelaskan satu dampak finance dari masing-masing jenis kesalahan.
5. Tulis memo 120–180 kata untuk atasan: tujuan, hasil utama, keterbatasan, dan rencana klarifikasi. Angka harus sesuai notebook. Kesimpulan tidak boleh menyatakan persetujuan kredit atau kepastian kejadian.
6. Catat prompt Gemini, hasil yang Anda periksa, koreksi, serta jalur Gemini/kode acuan. AiFn1 menambahkan daftar peran dan satu refleksi per anggota. AiFn2 mengerjakan sendiri.

**Jawaban dan memo Anda:**

**1. Tujuan Langkah 1 dan 2; masukan, target, prediksi**
*Langkah 1* membuka 24 catatan historis fiktif (H001–H024), menghitung rasio cicilan, dan memisahkan 18 contoh latih dari 6 contoh uji. *Langkah 2* meminta dua model belajar (`fit`) hanya dari 18 contoh latih.

| | Masukan (input) | Target historis (jawaban yang dipelajari) | Prediksi (keluaran) |
|---|---|---|---|
| Regresi | pendapatan_bulanan | pengeluaran_rutin_non_cicilan (Rp/bulan) | estimasi pengeluaran rutin (Rp juta/bulan) |
| Klasifikasi | rasio_persen saat pengajuan | terlambat_30hari_dalam_3bulan (0/1) | kategori Terlambat / Tidak terlambat >=30 hari |

ID dan kolom `bagian` tidak dipakai sebagai masukan. Kejadian terlambat juga tidak boleh jadi masukan, karena baru diketahui setelah pencairan, sedangkan saat pengajuan informasinya belum ada.

**2. Kenapa regresi dan kenapa klasifikasi**
Pengeluaran rutin berupa angka rupiah yang bisa bernilai berapa saja, jadi cocok dengan **regresi** yang menghasilkan angka. Keterlambatan berupa kategori, terjadi (1) atau tidak (0), jadi cocok dengan **klasifikasi** yang menghasilkan label.

**3. Input, hasil, dan perubahan**
- Awal: Rp8 juta dan Rp2,8 juta per bulan, rasio 35%, estimasi pengeluaran Rp4,87 juta per bulan, kategori Terlambat >=30 hari.
- Skenario: Rp10 juta dan Rp2,8 juta per bulan, rasio 28%, estimasi pengeluaran Rp5,98 juta per bulan, kategori Tidak terlambat >=30 hari.
- Yang berubah hanya pendapatan, dan cicilannya tetap. Rasio turun **7 poin persentase**, estimasi pengeluaran naik Rp1,11 juta, dan kategorinya berbalik. Kode acuan Langkah 5 memberi hasil yang sama.

**4. Contoh uji yang tidak cocok** (4 dari 6 cocok)
- **H023** (rasio 41%): diprediksi terlambat, padahal aktualnya tidak. Kesalahan jenis ini membuat pemohon yang sebenarnya lancar ditinjau berlebihan atau dirugikan, sehingga bank bisa kehilangan nasabah baik.
- **H024** (rasio 29%): diprediksi tidak terlambat, padahal aktualnya terlambat. Kesalahan jenis ini membuat risiko tidak terlihat, sehingga bank bisa kurang waspada dan menanggung tunggakan.

**5. Memo untuk atasan**
Tujuan analisis ini adalah mencoba dua model sederhana pada 24 data historis fiktif. Model belajar dari 18 contoh dan diuji pada 6 contoh. Untuk pemohon berpendapatan Rp8 juta dengan cicilan Rp2,8 juta per bulan, rasio cicilannya 35%. Model regresi memperkirakan pengeluaran rutin Rp4,87 juta per bulan, dan model klasifikasi memberi kategori "Terlambat >=30 hari". Pada skenario pendapatan Rp10 juta dengan cicilan tetap, rasio turun 7 poin persentase menjadi 28%, estimasi pengeluaran menjadi Rp5,98 juta, dan kategorinya berubah menjadi "Tidak terlambat". Pada data uji, kategori cocok 4 dari 6 contoh. H023 dan H024 salah prediksi, dan estimasi pengeluaran masih meleset sekitar Rp0,1–0,3 juta. Keterbatasannya: datanya kecil dan fiktif, tiap model hanya memakai satu masukan, dan kategori bukan probabilitas atau kepastian kejadian. Rencana tindak lanjut: minta bukti pendapatan, konfirmasi pengeluaran rutin aktual, tanggungan, dan kewajiban lain pemohon, lalu uji model dengan data yang lebih banyak. Hasil ini bukan keputusan persetujuan kredit.

**6. Deklarasi bantuan AI**
- *Gemini:* belum digunakan. Langkah 5 dikerjakan memakai **kode acuan** dari dosen (`hasil_acuan_p4`), dan sel kode Gemini dibiarkan kosong.
- *AI lain:* Claude (Anthropic), untuk membantu menjalankan notebook dan membuat draf jawaban.
- *Pemeriksaan sendiri:* menjalankan ulang semua sel di Colab, menghitung manual rasio 2,8 ÷ 8 = 35% dan 2,8 ÷ 10 = 28% (selisih 7 poin persentase), mencocokkan estimasi Rp4,87 juta dan Rp5,98 juta serta hasil 4 dari 6 cocok dengan tabel Langkah 3 dan 4, lalu menulis ulang jawaban dengan kata-kata sendiri.
- AiFn2: dikerjakan sendiri.

### Checklist pengumpulan
- [ ] Identitas, output, jawaban, memo, dan deklarasi AI tersimpan.
- [ ] Input dan satuan dibedakan dari hasil prediksi.
- [ ] Data uji tetap terpisah. Keterbatasan dan tindak lanjut dijelaskan.
- [ ] Tautan salinan pribadi/kelompok diberikan ke email dosen sebagai Viewer melalui LMS, atau file `.ipynb` dengan output jika berbagi dibatasi.

Tenggat dan penalti mengikuti instruksi LMS. Jika akses gagal, gunakan lembar cadangan pada Tugas P4, laporkan kendala, kemudian lengkapi eksekusi sesuai arahan dosen. Kuis dikerjakan individual tanpa materi dan AI.